In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
# from notebookutils import notebookutils
# Raw CMS files
BENE_PATH = "Files/raw/beneficiary"
INPATIENT_PATH = "Files/raw/inpatient"
OUTPATIENT_PATH = "Files/raw/outpatient"

# References
GEM_PATH = "Files/reference/gem/2018_I9gem.txt"

V22_MAPPING_PATH = "Files/reference/hcc/v22/mapping/*"
V28_MAPPING_PATH = "Files/reference/hcc/v28/mapping/*"

V22_MODEL_PATH = "Files/reference/hcc/v22/model/*"
V28_MODEL_PATH = "Files/reference/hcc/v28/model/*"



In [ ]:
for f in notebookutils.fs.ls(
    "Files/reference/hcc/v22/mapping"
):
    print(f.name)

files = notebookutils.fs.ls(
    "Files/reference/hcc/v28/mapping"
)

for f in files:
    print(f.name)

In [ ]:
V22_MAPPING_PATH = (
    "Files/reference/hcc/v22/mapping/*.csv"
)

df_v22_mapping_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(V22_MAPPING_PATH)
)

df_v22_mapping_raw.printSchema()

display(
    df_v22_mapping_raw.limit(20)
)

In [ ]:
V28_MAPPING_PATH = (
    "Files/reference/hcc/v28/mapping/*.csv"
)

df_v28_mapping_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(V28_MAPPING_PATH)
)

df_v28_mapping_raw.printSchema()

display(
    df_v28_mapping_raw.limit(20)
)

In [ ]:
import re

def sanitize_columns(df):
    safe_cols = []

    for c in df.columns:
        safe = c.strip().lower()

        # Replace spaces and all Delta-invalid characters
        safe = re.sub(r'[^a-zA-Z0-9_]', '_', safe)

        # Collapse multiple underscores
        safe = re.sub(r'_+', '_', safe)

        # Remove leading/trailing underscores
        safe = safe.strip('_')

        safe_cols.append(safe)

    return df.toDF(*safe_cols)

In [ ]:
df_v22_clean = sanitize_columns(df_v22_mapping_raw)
df_v28_clean = sanitize_columns(df_v28_mapping_raw)

In [ ]:
invalid_pattern = re.compile(r'[ ,;{}()\n\t=]')

for name in df_v22_clean.columns:
    assert not invalid_pattern.search(name), f"Invalid V22 column: {repr(name)}"

for name in df_v28_clean.columns:
    assert not invalid_pattern.search(name), f"Invalid V28 column: {repr(name)}"

print("All column names are Delta-safe.")

In [ ]:
(
    df_v22_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_hcc_v22_mapping_raw")
)

(
    df_v28_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_hcc_v28_mapping_raw")
)

In [ ]:
# Load main tabular datasets
df_bene_2008 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        f"{BENE_PATH}/DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv"
    )
)

df_bene_2009 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        f"{BENE_PATH}/DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv"
    )
)

df_bene_2010 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        f"{BENE_PATH}/DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv"
    )
)

df_inpatient = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        f"{INPATIENT_PATH}/DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv"
    )
)

df_outpatient = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        f"{OUTPATIENT_PATH}/DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv"
    )
)

df_gem_raw = spark.read.text(GEM_PATH)

In [ ]:
source_counts = {
    "beneficiary_2008": df_bene_2008.count(),
    "beneficiary_2009": df_bene_2009.count(),
    "beneficiary_2010": df_bene_2010.count(),
    "inpatient": df_inpatient.count(),
    "outpatient": df_outpatient.count(),
    "gem_lines": df_gem_raw.count()
}

for name, count in source_counts.items():
    print(f"{name:<20} {count:,}")

In [ ]:
print("=== Inpatient Schema ===")
df_inpatient.printSchema()

print("=== Outpatient Schema ===")
df_outpatient.printSchema()

print("=== Beneficiary 2010 Schema ===")
df_bene_2010.printSchema()

## Bronze Data Quality Validation

This section validates the structural integrity of the raw CMS DE-SynPUF
source data before any clinical transformation is performed.

Checks include:

- beneficiary identifier completeness and uniqueness,
- claim identifier completeness and uniqueness,
- claim-to-beneficiary referential integrity,
- service-date validity,
- payment-field validity,
- diagnosis availability,
- and source-level duplicate detection.

No ICD conversion or HCC assignment is performed in the Bronze layer.

In [ ]:
def beneficiary_quality(df, year):
    return (
        df.agg(
            F.count("*").alias("rows"),
            F.countDistinct("DESYNPUF_ID").alias("distinct_members"),

            F.sum(
                F.when(
                    F.col("DESYNPUF_ID").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_member_ids")
        )
        .withColumn(
            "year",
            F.lit(year)
        )
    )


bene_dq = (
    beneficiary_quality(df_bene_2008, 2008)
    .unionByName(
        beneficiary_quality(df_bene_2009, 2009)
    )
    .unionByName(
        beneficiary_quality(df_bene_2010, 2010)
    )
)

display(bene_dq)

In [ ]:
def claim_quality(df, source_name):

    return (
        df
        .withColumn(
            "claim_segment_key",
            F.concat_ws(
                "_",
                F.col("CLM_ID").cast("string"),
                F.col("SEGMENT").cast("string")
            )
        )
        .agg(
            F.count("*").alias("rows"),

            F.countDistinct(
                "claim_segment_key"
            ).alias("distinct_claim_segments"),

            F.sum(
                F.when(
                    F.col("CLM_ID").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_claim_ids"),

            F.sum(
                F.when(
                    F.col("DESYNPUF_ID").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_member_ids")
        )
        .withColumn(
            "source",
            F.lit(source_name)
        )
    )

In [ ]:
claim_dq = (
    claim_quality(
        df_inpatient,
        "INPATIENT"
    )
    .unionByName(
        claim_quality(
            df_outpatient,
            "OUTPATIENT"
        )
    )
)

display(claim_dq)

In [ ]:
# Convert the integer dates only for validation
def validate_claim_dates(df, source_name):

    temp = (
        df
        .withColumn(
            "from_date",
            F.to_date(
                F.col("CLM_FROM_DT").cast("string"),
                "yyyyMMdd"
            )
        )
        .withColumn(
            "thru_date",
            F.to_date(
                F.col("CLM_THRU_DT").cast("string"),
                "yyyyMMdd"
            )
        )
    )

    return (
        temp
        .agg(
            F.count("*").alias("rows"),

            F.sum(
                F.when(
                    F.col("from_date").isNull(),
                    1
                ).otherwise(0)
            ).alias("invalid_from_dates"),

            F.sum(
                F.when(
                    F.col("thru_date").isNull(),
                    1
                ).otherwise(0)
            ).alias("invalid_thru_dates"),

            F.sum(
                F.when(
                    F.col("thru_date")
                    <
                    F.col("from_date"),
                    1
                ).otherwise(0)
            ).alias("end_before_start")
        )
        .withColumn(
            "source",
            F.lit(source_name)
        )
    )

In [ ]:
date_dq = (
    validate_claim_dates(
        df_inpatient,
        "INPATIENT"
    )
    .unionByName(
        validate_claim_dates(
            df_outpatient,
            "OUTPATIENT"
        )
    )
)

display(date_dq)

In [ ]:
# Negative payments can exist in claims datasets because of adjustments, so we should profile them, not automatically classify them as invalid.
def payment_profile(df, source_name):

    return (
        df
        .agg(
            F.count("*").alias("rows"),

            F.sum(
                F.when(
                    F.col("CLM_PMT_AMT").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_payment_amount"),

            F.sum(
                F.when(
                    F.col("CLM_PMT_AMT") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_payment_amount"),

            F.round(
                F.min("CLM_PMT_AMT"),
                2
            ).alias("min_payment"),

            F.round(
                F.avg("CLM_PMT_AMT"),
                2
            ).alias("avg_payment"),

            F.round(
                F.max("CLM_PMT_AMT"),
                2
            ).alias("max_payment")
        )
        .withColumn(
            "source",
            F.lit(source_name)
        )
    )

In [ ]:
payment_dq = (
    payment_profile(
        df_inpatient,
        "INPATIENT"
    )
    .unionByName(
        payment_profile(
            df_outpatient,
            "OUTPATIENT"
        )
    )
)

display(payment_dq)

In [ ]:
diagnosis_cols = [
    f"ICD9_DGNS_CD_{i}"
    for i in range(1, 11)
]

def diagnosis_profile(df, source_name):

    return (
        df
        .withColumn(
            "diagnosis_count",
            sum(
                F.when(
                    F.col(c).isNotNull(),
                    1
                ).otherwise(0)
                for c in diagnosis_cols
            )
        )
        .agg(
            F.count("*")
            .alias("rows"),

            F.sum(
                F.when(
                    F.col("ICD9_DGNS_CD_1").isNotNull(),
                    1
                ).otherwise(0)
            ).alias("claims_with_primary_diagnosis"),

            F.sum(
                F.when(
                    F.col("diagnosis_count") > 0,
                    1
                ).otherwise(0)
            ).alias("claims_with_any_diagnosis"),

            F.round(
                F.avg("diagnosis_count"),
                2
            ).alias("avg_diagnoses_per_claim")
        )
        .withColumn(
            "source",
            F.lit(source_name)
        )
    )

In [ ]:
diagnosis_dq = (
    diagnosis_profile(
        df_inpatient,
        "INPATIENT"
    )
    .unionByName(
        diagnosis_profile(
            df_outpatient,
            "OUTPATIENT"
        )
    )
)

display(diagnosis_dq)

In [ ]:
df_all_beneficiary_ids = (
    df_bene_2008
    .select("DESYNPUF_ID")
    .union(
        df_bene_2009
        .select("DESYNPUF_ID")
    )
    .union(
        df_bene_2010
        .select("DESYNPUF_ID")
    )
    .distinct()
)
inpatient_orphans = (
    df_inpatient
    .select("DESYNPUF_ID")
    .distinct()
    .join(
        df_all_beneficiary_ids,
        "DESYNPUF_ID",
        "left_anti"
    )
    .count()
)
outpatient_orphans = (
    df_outpatient
    .select("DESYNPUF_ID")
    .distinct()
    .join(
        df_all_beneficiary_ids,
        "DESYNPUF_ID",
        "left_anti"
    )
    .count()
)

In [ ]:
print(
    "Inpatient orphan beneficiaries:",
    inpatient_orphans
)

print(
    "Outpatient orphan beneficiaries:",
    outpatient_orphans
)

In [ ]:
# missing dates from truly invalid formatted dates.
def detailed_date_profile(df, source_name):

    temp = (
        df
        .withColumn(
            "parsed_from_date",
            F.to_date(
                F.col("CLM_FROM_DT").cast("string"),
                "yyyyMMdd"
            )
        )
        .withColumn(
            "parsed_thru_date",
            F.to_date(
                F.col("CLM_THRU_DT").cast("string"),
                "yyyyMMdd"
            )
        )
    )

    return (
        temp
        .agg(
            F.count("*").alias("rows"),

            F.sum(
                F.when(
                    F.col("CLM_FROM_DT").isNull(),
                    1
                ).otherwise(0)
            ).alias("missing_from_date"),

            F.sum(
                F.when(
                    F.col("CLM_FROM_DT").isNotNull()
                    &
                    F.col("parsed_from_date").isNull(),
                    1
                ).otherwise(0)
            ).alias("malformed_from_date"),

            F.sum(
                F.when(
                    F.col("CLM_THRU_DT").isNull(),
                    1
                ).otherwise(0)
            ).alias("missing_thru_date"),

            F.sum(
                F.when(
                    F.col("CLM_THRU_DT").isNotNull()
                    &
                    F.col("parsed_thru_date").isNull(),
                    1
                ).otherwise(0)
            ).alias("malformed_thru_date")
        )
        .withColumn(
            "source",
            F.lit(source_name)
        )
    )

date_detail = (
    detailed_date_profile(
        df_inpatient,
        "INPATIENT"
    )
    .unionByName(
        detailed_date_profile(
            df_outpatient,
            "OUTPATIENT"
        )
    )
)

display(date_detail)

In [ ]:
(
    df_bene_2008
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_beneficiary_2008")
)

(
    df_bene_2009
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_beneficiary_2009")
)

(
    df_bene_2010
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_beneficiary_2010")
)

(
    df_inpatient
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_inpatient_claims")
)

(
    df_outpatient
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_outpatient_claims")
)

(
    df_gem_raw
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_icd9_icd10_gem_raw")
)

In [ ]:
bronze_tables = [
    "bronze_beneficiary_2008",
    "bronze_beneficiary_2009",
    "bronze_beneficiary_2010",
    "bronze_inpatient_claims",
    "bronze_outpatient_claims",
    "bronze_icd9_icd10_gem_raw"
]

for table_name in bronze_tables:
    print(
        f"{table_name:<35}",
        f"{spark.table(table_name).count():,}"
    )

In [ ]:
print("=== V22 MODEL FILES ===")

for f in notebookutils.fs.ls(
    "Files/reference/hcc/v22/model"
):
    print(f.name)


print("=== V28 MODEL FILES ===")

for f in notebookutils.fs.ls(
    "Files/reference/hcc/v28/model"
):
    print(f.name)




In [ ]:
for f in notebookutils.fs.ls(
    "Files/reference/hcc/v22/model"
):
    if f.name in [
        "C2214O5P",
        "V22H79H1.TXT",
        "V2217O1M.TXT",
        "V22I0ED2.TXT"
    ]:
        print(f.name, f.size)


for f in notebookutils.fs.ls(
    "Files/reference/hcc/v28/model"
):
    if f.name in [
        "C2824T2N",
        "V28115H1.TXT",
        "V2824T1M.TXT",
        "V28I0ED2.TXT"
    ]:
        print(f.name, f.size)

In [ ]:
def inspect_model_file(path, keywords):
    text = notebookutils.fs.head(path, 50000)

    for line in text.splitlines():
        if any(
            keyword.lower() in line.lower()
            for keyword in keywords
        ):
            print(line)

keywords = [
    "C2214O5P",
    "C2824T2N",
    "COEF",
    "SCORE",
    "HCC",
    "COMMUNITY"
]

In [ ]:
print("=== V22 SCORE PROGRAM ===")

inspect_model_file(
    "Files/reference/hcc/v22/model/V2217O1P.TXT",
    keywords
)

print("=== V22 MODEL MACRO ===")

inspect_model_file(
    "Files/reference/hcc/v22/model/V2217O1M.TXT",
    keywords
)

print("=== V28 COEFFICIENT TEXT ===")

inspect_model_file(
    "Files/reference/hcc/v28/model/C2824T2N.TXT",
    keywords
)

print("=== V28 MODEL MACRO ===")

inspect_model_file(
    "Files/reference/hcc/v28/model/V2824T1M.TXT",
    keywords
)